In [1]:
%%configure -f
{
    "conf":{
        "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        "spark.sql.catalog.spark_catalog": "org.apache.iceberg.spark.SparkSessionCatalog",
        "spark.sql.catalog.spark_catalog.type": "glue",
        "spark.sql.catalog.spark_catalog.glue.id": "889415020100",
        "spark.sql.catalog.spark_catalog.warehouse" :"s3://aep-datalake-apps-dev/emr/warehouse/nonvee-lg-tx",
        "spark.sql.session.timeZone": "America/New_York",   
        "spark.driver.extraJavaOptions": "-Duser.timezone=America/New_York",
        "spark.executor.extraJavaOptions": "-Duser.timezone=America/New_York",
        "spark.driver.cores": "4",
        "spark.driver.memory": "8g",
        "spark.driver.maxResultSize": "16g",
        "spark.executor.cores": "1",
        "spark.executor.memory": "4g"
        
    }
}

In [ ]:
spark.sql("SHOW CATALOGS").show()

In [ ]:
df = spark.sql("""
select * from usage_nonvee.reading_ivl_nonvee_tx limit 10""")
df.show()

In [2]:
from pyspark.sql import functions as F
import logging
import sys

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
8,application_1774866862669_0018,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
df = spark.read.parquet("s3://aep-dl-util-consume-ami-nonvee-dev/intervals/iceberg_catalog/usage_nonvee/reading_ivl_nonvee_tx_ice/") \
        .filter(F.col("aep_usage_dt") == "2026-03-25").limit(10)

df.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
[UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually.
Traceback (most recent call last):
  File "/mnt2/yarn/usercache/livy/appcache/application_1774866862669_0018/container_1774866862669_0018_01_000001/pyspark.zip/pyspark/sql/readwriter.py", line 307, in load
    return self._df(self._jreader.load(path))
  File "/mnt2/yarn/usercache/livy/appcache/application_1774866862669_0018/container_1774866862669_0018_01_000001/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
  File "/mnt2/yarn/usercache/livy/appcache/application_1774866862669_0018/container_1774866862669_0018_01_000001/pyspark.zip/pyspark/errors/exceptions/captured.py", line 185, in deco
    raise converted from None
pyspark.errors.exceptions.captured.AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually.



In [ ]:
from pyspark.sql import functions as F
import logging
# import sys


# spark.sparkContext.setLogLevel("ERROR")


# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s | %(levelname)s | %(message)s",
#     handlers=[logging.StreamHandler(sys.stdout)]
# )

# logger = logging.getLogger("validation_logger")

# logger.info("===== START VALIDATION JOB =====")


dev_df = spark.read.parquet("s3")
prod_df = spark.read.parquet("s3")


current_dt = F.current_date()

dev_df = dev_df.filter(F.to_date("aep_usage_dt") == current_dt)
prod_df = prod_df.filter(F.to_date("aep_usage_dt") == current_dt)

# ==========================================
# 1. COUNT CHECK
# ==========================================
dev_count = dev_df.count()
prod_count = prod_df.count()

logger.info("===== COUNT CHECK =====")
logger.info(f"DEV  count: {dev_count}")
logger.info(f"PROD count: {prod_count}")

# ==========================================
# 2. PICK 2 SERIAL NUMBERS 
# ==========================================
sample_serials = (
    dev_df.select("serialnumber")
    .distinct()
    .limit(2)
    .rdd.flatMap(lambda x: x)
    .collect()
)

logger.info(f"Sample serialnumbers: {sample_serials}")

dev_df = dev_df.filter(F.col("serialnumber").isin(sample_serials))
prod_df = prod_df.filter(F.col("serialnumber").isin(sample_serials))

# ==========================================
# 3. JOIN CONDITIONS
# ==========================================
join_cond = (
    (dev_df.aep_usage_dt == prod_df.aep_usage_dt) &
    (dev_df.aep_meter_bucket == prod_df.aep_meter_bucket) &
    (dev_df.serialnumber == prod_df.serialnumber) &
    (dev_df.aep_raw_uom == prod_df.aep_raw_uom) &
    (dev_df.endtimeperiod == prod_df.endtimeperiod)
)

joined_df = dev_df.alias("t").join(
    prod_df.alias("s"),
    on=join_cond,
    how="outer"
)

# ==========================================
# 4. COMPARE ALL COLUMNS
# ==========================================
cols = dev_df.columns

compare_expr = [
    F.when(
        (F.col(f"t.{c}") != F.col(f"s.{c}")) |
        (F.col(f"t.{c}").isNull() != F.col(f"s.{c}").isNull()),
        F.lit("Mismatch")
    ).otherwise(F.lit("Match")).alias(f"{c}_check")
    for c in cols
]

diff_df = joined_df.select(
    *[F.col(f"t.{c}").alias(f"dev_{c}") for c in cols],
    *[F.col(f"s.{c}").alias(f"prod_{c}") for c in cols],
    *compare_expr
)

# ==========================================
# RESULTS
# ==========================================
mismatch_count = mismatch_df.count()

logger.info("===== MISMATCH COUNT =====")
logger.info(f"Mismatched rows: {mismatch_count}")

if mismatch_count > 0:
    mismatch_df.show(20, False)


logger.info("===== END VALIDATION JOB =====")